# Preprocess raw inputs before IMSTE

This notebook shows the built-in text, image, and graph presets, plus a caller-supplied scikit-learn transformer. Each preprocessor turns raw examples into feature vectors before IMSTE fits 2D coordinates. Labels are used only to color the plots.

Install text/image and plotting dependencies with `python -m pip install "imste[preprocessing,notebook]"`; install NSPPK separately with `python -m pip install git+https://github.com/Xfcosta/NSPPK.git`. The first text or image fit downloads the Google EmbeddingGemma 2 weights if needed.

In [1]:
import gc
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw
from sklearn.feature_extraction.text import TfidfVectorizer

from imste import IteratedMinimumSpanningTreeEmbedder
from imste.visualization import plot_embedding

def make_mapper():
    return IteratedMinimumSpanningTreeEmbedder(
        n_msts=3, n_epochs=80, random_state=42, device="auto"
    )

def show_embedding(embedding, labels, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_embedding(embedding, labels, point_size=24, ax=ax)
    ax.set_title(title)
    display(fig)
    plt.close(fig)

## Text with EmbeddingGemma 2

Pass a sequence of strings to the `text` preset.

In [2]:
variants = [
    "explores a quiet forest trail",
    "rests beside a mountain stream",
    "moves through a field at sunrise",
    "looks for food near the old trees",
    "crosses a meadow under cloudy skies",
    "returns home after a long journey",
    "finds shelter during the rain",
    "watches the changing seasons",
]
topics = ["A red fox", "A sailing boat", "A city tram"]
texts = [f"{topic} {variant}." for topic in topics for variant in variants]
text_labels = np.repeat(topics, len(variants))

text_mapper = make_mapper().preprocessor("text", batch_size=16)
text_embedding = text_mapper.fit_transform(texts)
show_embedding(text_embedding, text_labels, "Text embedding")

del text_mapper
gc.collect()

AttributeError: 'IteratedMinimumSpanningTreeEmbedder' object has no attribute 'preprocessor'

## Images with EmbeddingGemma 2

This example draws small synthetic images in memory; no image dataset download is needed.

In [ ]:
image_labels = np.repeat(["circle", "square", "triangle"], 8)
images = []
for class_index, color in enumerate(("#d9473f", "#3478c8", "#31945c")):
    for variation in range(8):
        image = Image.new("RGB", (112, 112), "white")
        draw = ImageDraw.Draw(image)
        offset = variation * 2
        bounds = (20 + offset, 20 + offset, 92 - offset, 92 - offset)
        if class_index == 0:
            draw.ellipse(bounds, fill=color)
        elif class_index == 1:
            draw.rectangle(bounds, fill=color)
        else:
            draw.polygon([(56, 12 + offset), (100 - offset, 96 - offset), (12 + offset, 96 - offset)], fill=color)
        images.append(image)

image_mapper = make_mapper().preprocessor("image", batch_size=8)
image_embedding = image_mapper.fit_transform(images)
show_embedding(image_embedding, image_labels, "Image embedding")

del image_mapper
gc.collect()

## Graphs with NSPPK

The graph preset takes graph-level NetworkX objects and encodes each graph as a feature vector.

In [ ]:
graph_labels = np.repeat(["path", "cycle", "star"], 8)
graphs = []
for class_index in range(3):
    for variation in range(8):
        n_nodes = 8 + variation % 4
        if class_index == 0:
            graph = nx.path_graph(n_nodes)
        elif class_index == 1:
            graph = nx.cycle_graph(n_nodes)
        else:
            graph = nx.star_graph(n_nodes - 1)
        graphs.append(graph)

graph_mapper = make_mapper().preprocessor(
    "graph", radius=2, distance=3, nbits=12, parallel=False, dense=True
)
graph_embedding = graph_mapper.fit_transform(graphs)
show_embedding(graph_embedding, graph_labels, "Graph embedding")

## Caller-supplied scikit-learn transformer

Any transformer implementing `fit` and `transform` can be attached. Sparse output, such as TF-IDF, is supported by IMSTE.

In [ ]:
tfidf_mapper = make_mapper().preprocessor(
    TfidfVectorizer(ngram_range=(1, 2))
)
tfidf_embedding = tfidf_mapper.fit_transform(texts)
show_embedding(tfidf_embedding, text_labels, "TF-IDF + IMSTE")